# 08. Dense Retrieval & Generation Evaluation Benchmark

This notebook demonstrates the quantitative evaluation framework used to score the baseline Dense Vector Search strategy across retrieval metrics (MRR, nDCG, Keyword Coverage) and LLM-as-a-Judge answer quality (`src/step8_evals_dense.py`).

### Key Demonstration Steps:
1. **Benchmark Test Loading**: Parsing synthetic test items from `tests.jsonl` into typed `TestItem` structures.
2. **Single-Item Evaluation Workflow**: Running `get_chunks_from_db`, `evaluate_retrieval`, and `evaluate_answer` sequentially on a sample test query.
3. **Evaluation Metric Aggregation**: Inspecting structured result DataFrames and calculating mean Accuracy, Completeness, and Source Hit Rates.

In [1]:
import sys
import os
import pandas as pd

# Add project root to path for modular imports
sys.path.append("..")

from src.step8_evals_dense import (
    load_test_dataset, 
    run_single_eval, 
    TestItem,
    PROCESSED_DATA_DIR,
    RESULTS_DIR
)

## Step 1: Benchmark Dataset Loading

Load evaluation test items from `tests.jsonl` and inspect the dataset balance across categories.

In [2]:
test_file = PROCESSED_DATA_DIR / "tests.jsonl"
tests = load_test_dataset(test_file)

# Convert test list to DataFrame for quick categorical breakdown
df_tests = pd.DataFrame([vars(t) for t in tests])

print("\n--- Categorical Distribution of Benchmark Tests ---")
print(df_tests['category'].value_counts())

Loaded 274 evaluation tests from 'tests.jsonl'.

--- Categorical Distribution of Benchmark Tests ---
category
cause_and_effect_praxis                  67
explicit_definition_or_classification    56
textual_quote_grounding                  52
named_entities_and_location              51
exact_analogy_or_parable                 48
Name: count, dtype: int64


## Step 2: Run Dense Retrieval Single-Item Evaluation

Execute `run_single_eval` on the first benchmark test item to observe retrieval scoring (MRR, nDCG) and LLM judge scoring (Accuracy, Completeness, Relevance).

In [3]:
sample_test = tests[0]

print(f"Executing Dense Evaluation for Test Item:")
print(f"• Question    : {sample_test.question}")
print(f"• Category    : {sample_test.category}")
print(f"• Target File : {sample_test.source_file}\n")

eval_result = run_single_eval(sample_test)

print("--- Evaluation Output Payload ---")
print(f"Source Found     : {eval_result['source_found']}")
print(f"MRR / nDCG       : {eval_result['mrr']:.2f} / {eval_result['ndcg']:.2f}")
print(f"Keyword Coverage : {eval_result['keyword_coverage']*100:.1f}% ({eval_result['keywords_found']})")
print(f"Accuracy Score   : {eval_result['accuracy']} / 5.0")
print(f"Completeness     : {eval_result['completeness']} / 5.0")
print(f"Relevance        : {eval_result['relevance']} / 5.0")
print(f"Judge Feedback   : {eval_result['judge_feedback']}")

Executing Dense Evaluation for Test Item:
• Question    : What does the letter dictated by Shriji Maharaj emphasize about attaining a human birth in Bharat-khand?
• Category    : exact_analogy_or_parable
• Target File : Gadhada_I_1.md

--- Evaluation Output Payload ---
Source Found     : True
MRR / nDCG       : 0.25 / 0.25
Keyword Coverage : 100.0% (5/5)
Accuracy Score   : 5.0 / 5.0
Completeness     : 5.0 / 5.0
Relevance        : 5.0 / 5.0
Judge Feedback   : The Generated Answer accurately captures the key points of the Reference Answer, emphasizing the rarity and significance of human birth in Bharat-khand, the comparison to chintamani, and the longing of deities for this opportunity. It also expands on the implications for spiritual growth and liberation, aligning well with the original message.


## Step 3: Result Persistence & Path Verification

Verify configured output paths for saving dense evaluation CSV and JSON files in `RESULTS_DIR`.

In [4]:
csv_output = RESULTS_DIR / "eval_results_dense.csv"
json_output = RESULTS_DIR / "eval_results_dense.json"

print(f"Target CSV Output Path  : {csv_output}")
print(f"Target JSON Output Path : {json_output}")

Target CSV Output Path  : C:\Users\Lenovo\projects\Active Vachanamrut RAG project\results\eval_results_dense.csv
Target JSON Output Path : C:\Users\Lenovo\projects\Active Vachanamrut RAG project\results\eval_results_dense.json
